# Reviews.csv — Data Exploration

Systematic exploration of `reviews.csv`:
structural profile → null/missingness pass → duplicate pass → referential integrity →
value-range sanity → candidate questions.

Findings are written up separately in `docs/reviews_data_exploration_report.pdf`. This notebook is the working that supports the words in the pdf — every claim in that report traces back to a cell here.

## Setup

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
data_path = Path("../data/raw")
reviews = pd.read_csv(data_path / "reviews.csv")
reviews.shape

(6327, 23)

## 1. Structural Profile

row/column count, column names, and whether pandas inferred sensible
dtypes.

In [3]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 6327 entries, 0 to 6326
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   s.no                 6327 non-null   int64  
 1   helpfulVoteCount     6327 non-null   int64  
 2   images/0             516 non-null    str    
 3   images/1             212 non-null    str    
 4   images/2             98 non-null     str    
 5   images/3             49 non-null     str    
 6   images/4             26 non-null     str    
 7   images/5             14 non-null     str    
 8   images/6             7 non-null      str    
 9   images/7             5 non-null      str    
 10  productASIN          6327 non-null   str    
 11  productVariant       5836 non-null   str    
 12  rating               6320 non-null   float64
 13  reviewID             6327 non-null   str    
 14  reviewMetadata       6296 non-null   str    
 15  reviewPosition       6327 non-null   int64  
 16 

In [4]:
list(reviews.columns)

['s.no',
 'helpfulVoteCount',
 'images/0',
 'images/1',
 'images/2',
 'images/3',
 'images/4',
 'images/5',
 'images/6',
 'images/7',
 'productASIN',
 'productVariant',
 'rating',
 'reviewID',
 'reviewMetadata',
 'reviewPosition',
 'reviewText',
 'reviewTitle',
 'reviewURL',
 'verifiedPurchase',
 'videos/0',
 'cleaned_review_text',
 'sentiment_score']

**Note:** all dtypes look correct at face value (`rating`/`sentiment_score` as float,
`verifiedPurchase` as bool, ids/text as str). No silent-mistype issues like a price column
read as text — worth confirming again once `products.csv` is profiled, since that file has
more numeric-looking columns at higher risk of this.

## 2. Null / Missingness Pass

Percentage of nulls. Also checking whether nulls are random or
*structurally expected* (e.g. a review with no photo attached is not "missing" data in the
same sense as a rating that should exist but doesn't).

In [5]:
null_pct = (reviews.isnull().mean() * 100).sort_values(ascending=False)
null_pct[null_pct > 0]

images/7               99.920974
images/6               99.889363
images/5               99.778726
images/4               99.589063
videos/0               99.462621
images/3               99.225541
images/2               98.451083
images/1               96.649281
images/0               91.844476
productVariant          7.760392
cleaned_review_text     0.616406
reviewText              0.600601
reviewMetadata          0.489964
reviewTitle             0.221274
rating                  0.110637
dtype: float64

**Image/video columns (`images/0`–`images/7`, `videos/0`):** null rate climbs from ~92%
(`images/0`) to ~99.9% (`images/7`) — expected shape, since most reviews attach zero photos
and a shrinking few attach more. Not a data quality problem; it's the natural distribution of
"how many photos does a reviewer bother to attach." Confirmed every populated link resolves to
a real-looking URL (checked below) rather than being garbage.

In [6]:
image_cols = [c for c in reviews.columns if c.startswith("images/")]
all_images = reviews[image_cols].stack().dropna()
ends_jpg = all_images.str.lower().str.endswith(".jpg")

print(f"Total non-null image links: {len(all_images)}")
print(f"Ending .jpg:               {ends_jpg.sum()} ({ends_jpg.mean()*100:.1f}%)")
print(f"NOT ending .jpg:           {(~ends_jpg).sum()}")

Total non-null image links: 927
Ending .jpg:               927 (100.0%)
NOT ending .jpg:           0


All 927 populated image links end in `.jpg` — clean, no truncation issue on this
column.

In [7]:
videos = reviews["videos/0"].dropna()
ends_mp4 = videos.str.lower().str.endswith(".mp4")

print(f"Total non-null video links: {len(videos)}")
print(f"Ending .mp4:                {ends_mp4.sum()}")
print(f"NOT ending .mp4 (truncated):{(~ends_mp4).sum()}")
print()
print("Truncated video links (real quality issue, not just missing):")
print(videos[~ends_mp4].tolist())

Total non-null video links: 34
Ending .mp4:                31
NOT ending .mp4 (truncated):3

Truncated video links (real quality issue, not just missing):
['https://m.media-amazon.com/images/I/E1O', 'https://m.media-amazon.com/images/I/E1as', 'https://m.media-amazon.com/images/I/H1xtC']


**Real finding, distinct from "missing":** 3 of 34 populated `videos/0` links (≈9%) are
**truncated URLs**, not nulls — e.g. `.../E1O` instead of a full `.mp4` link. This is
corruption within populated data, not absence of data, and needs its own handling decision in
the Transformer (drop the row's video field, or flag it) separate from the ~99% that are legitimately null.

In [8]:
reviewMetadata_null = reviews["reviewMetadata"].isna()
reviews.loc[reviewMetadata_null, ["reviewID", "reviewText", "rating"]].head(10)

,reviewID,reviewText,rating
147,RD6E7ZGCFVB9E,NaN,NaN
1226,R1CISIPONTZZ8Q,NaN,5.0
1877,R27TG77CVE8SXM,NaN,NaN
2008,R2JYNJ2K2LNDOZ,NaN,5.0
2244,R1WFL6RRGQOVMW,NaN,5.0
2560,R8VK2VT929E3S,NaN,4.0
3100,R3R68SQ0595KTY,NaN,5.0
3373,RPGZ52Y3H4SS9,NaN,5.0
3407,RO3PAZJZVEOQG,NaN,NaN
3414,R26JVEOW0AXJ3L,NaN,5.0


`reviewMetadata` (location + date string) is null for ~0.5% of rows. Spot check above:
these rows still have real `reviewText`/`rating` — this looks like an isolated scraping gap on
that one field, not a sign the whole row is bad. Safe to keep the row, `reviewMetadata` may be treated later as nullable.

In [9]:
# Does a null rating correlate with anything specific, or is it scattered?
reviews.loc[reviews["rating"].isna(), ["reviewID", "reviewText", "sentiment_score"]]

,reviewID,reviewText,sentiment_score
147,RD6E7ZGCFVB9E,NaN,0.0
1877,R27TG77CVE8SXM,NaN,0.0
3407,RO3PAZJZVEOQG,NaN,0.0
5368,R3MR3VMXYQNLAF,NaN,0.0
5419,R2XSHH4UXPNZ4M,NaN,0.0
5461,R3ARJZISNG1OKX,NaN,0.0
5632,R2R4Q07GPEXK2J,NaN,0.0


**Pattern found:** all 7 rows with a null `rating` also have null `reviewText`, and all 7
have `sentiment_score == 0.0` exactly. Worth treating as random — worth checking next whether `sentiment_score = 0.0` is a genuine "neutral" computed score, or simply a default value the scraper/pipeline fills in when there's no text to score at all.

In [10]:
# Is sentiment_score == 0.0 specifically tied to missing text, or does 0.0 also occur
# on rows that DO have text (which would mean 0.0 is a real neutral score, not a default)?
zero_sentiment = reviews[reviews["sentiment_score"] == 0.0]
print(f"Rows with sentiment_score == 0.0: {len(zero_sentiment)}")
print(f"Of those, rows with null reviewText: {zero_sentiment['reviewText'].isna().sum()}")
print(f"Of those, rows WITH real reviewText: {zero_sentiment['reviewText'].notna().sum()}")

Rows with sentiment_score == 0.0: 251
Of those, rows with null reviewText: 38
Of those, rows WITH real reviewText: 213


**Verified below — the finding changes**: `0.0` occurs on 251 rows total, but only 38 of those have null `reviewText` — the other 213 have real review text and still score exactly `0.0`. So `0.0` is **not purely a null-text default** as first suspected; it's more likely a genuine possible output of the sentiment model that happens to coincide with the default value. This means the earlier hypothesis (treat `0.0` as "unknown" rather than "neutral") does not hold as a blanket rule — the 38 null-text rows are still worth flagging as likely-default, but the other 213 should be trusted as real scores. Worth reflecting this nuance in the Transformer design later.

In [11]:
# reviewText null but cleaned_review_text / sentiment_score still present — a mismatch
# between the "raw" and "cleaned" pipeline for the same review
mismatch = reviews["reviewText"].isna() & reviews["sentiment_score"].notna()
print(f"reviewText null but sentiment_score present: {mismatch.sum()}")
reviews.loc[mismatch, ["reviewID", "reviewText", "cleaned_review_text", "sentiment_score"]].head()

reviewText null but sentiment_score present: 38


,reviewID,reviewText,cleaned_review_text,sentiment_score
147,RD6E7ZGCFVB9E,NaN,NaN,0.0
1226,R1CISIPONTZZ8Q,NaN,NaN,0.0
1310,R308S4623OL56U,NaN,NaN,0.0
1520,R3SG5UQGW1LIES,NaN,NaN,0.0
1558,RLDZEV33GV4L7,NaN,NaN,0.0


## 3. Duplicate Pass

Full-row duplicates, and — the one that actually matters for schema design — key
duplicates on `reviewID`.

In [12]:
print("Full-row duplicates:", reviews.duplicated().sum())
print("reviewID duplicates: ", reviews["reviewID"].duplicated().sum())
print("reviewID is unique:  ", reviews["reviewID"].is_unique)

Full-row duplicates: 0
reviewID duplicates:  0
reviewID is unique:   True


`reviewID` is a clean, fully unique key — 6,327 rows, 6,327 distinct ids, zero
duplicates. Safe to use as the natural primary key / `_id` source for a `reviews` collection.

## 4. Referential Integrity (against `products.csv`)

Deferred to the joint exploration once `products.csv` is profiled.

In [13]:
reviews["productASIN"].nunique()  # distinct products referenced by at least one review

700

## 5. Value-Range / Sanity Pass

Numeric ranges, categorical value counts, and a deliberate spot-check of free text — plus
checking computed stats against whatever the dataset's own description claims, since those
can silently disagree.

In [14]:
reviews["rating"].describe()

count    6320.000000
mean        4.533228
std         0.855710
min         1.000000
25%         4.000000
50%         5.000000
75%         5.000000
max         5.000000
Name: rating, dtype: float64

`rating` ranges 1.0–5.0 as expected, always a whole number in practice (no genuine half
-star values observed) despite being stored as float64 — That will need casting in the Transformer design.

In [15]:
reviews["verifiedPurchase"].value_counts(dropna=False)

verifiedPurchase
True     6167
False     160
Name: count, dtype: int64

In [16]:
reviews["sentiment_score"].describe()

count    6327.000000
mean        0.307545
std         0.218020
min        -1.000000
25%         0.168568
50%         0.300000
75%         0.450000
max         1.000000
Name: sentiment_score, dtype: float64

**Discrepancy from the dataset's own listing:** the Kaggle description states
`sentiment_score` ranges 0–1, but the actual computed range here is **-1.0 to 1.0**. Which worth flagging and revising other things where needed.

In [17]:
reviews["reviewPosition"].value_counts().sort_index()

reviewPosition
1     696
2     668
3     653
4     639
5     625
6     619
7     610
8     609
9     602
10    606
Name: count, dtype: int64

`reviewPosition` is a clean 1–10 ranking (position of the review within its product's
review list), And with manual verification (looking myself to the review on amazon website)found that these positions is not quite correct and may not be needing this column anymore since its stale data.

In [18]:
# Spot-check free text directly — automated null/type checks won't catch encoding
# issues or non-English content mixed into a supposedly uniform field.
reviews[["reviewText", "cleaned_review_text", "rating", "sentiment_score"]].sample(5, random_state=3)

,reviewText,cleaned_review_text,rating,sentiment_score
2275,I use it every moring I have to bike to school...,use every moring bike school everyday really h...,5.0,0.313675
602,Purchased for my boyfriend. He loved it! The c...,purchased boyfriend loved color great size per...,5.0,0.437500
5161,The most comfortable shoes ever!,comfortable shoe ever,5.0,0.400000
4740,"Pretty, well-made, but not figure flattering l...",pretty wellmade figure flattering like photogr...,3.0,0.138300
3251,"Has had these for about a year now, and only o...",year one started rip besides job,4.0,0.000000


**Real finding from manual spot-checking (not caught by automated stats):** at least one
row contains a **non-English review** (Hindi) where `cleaned_review_text` is null despite
`reviewText` having real content — meaning the cleaning/sentiment pipeline that produced this
dataset likely only processes English text, silently leaving `cleaned_review_text` null and
`sentiment_score` at a default (`0.0`) for non-English rows rather than actually scoring them.
This directly connects to the `sentiment_score == 0.0` investigation above — it's likely the
same root cause (no clean text to score) showing up two different ways: missing text entirely,
or non-English text the cleaning step couldn't handle.

In [19]:
reviews.loc[
    reviews["reviewText"].notna() & reviews["cleaned_review_text"].isna(),
    ["reviewID", "reviewText", "reviewTitle", "reviewMetadata", "rating", "sentiment_score"]
]

,reviewID,reviewText,reviewTitle,reviewMetadata,rating,sentiment_score
5391,R1XPW9TTLHNW3K,3 महीने होने को है। आपने फटी हुई साड़ी भेज दी।...,आफत से कम नहीं ये साड़ी,"Reviewed in India on February 11, 2025",1.0,0.0


## 6. Column-Specific Notes

Short, structured per-column notes for the columns not already covered above in a dedicated
check.

In [20]:
reviews["productVariant"].dropna().sample(10, random_state=2)

3385                 Size: 6-12.5Color: Navy, Bright Navy
2314                       Height: 6'2"Weight: 225-230 lb
3469    Size: 10.5Color: Core Black/Core Black/Core Black
3472                        Size: 8Color: White/White/Gum
631                               Size: SmallColor: Black
5318           Size: 11-11.5 Women/9.5-10 MenColor: Taupe
1586    Size: XX-LargeColor: Navy BlueHeight: 5'7"Weig...
3681                      Size: 10.5Color: 01 Light/Brown
1046    Special Size: Big & TallSize: 50W x 32L Big Ta...
4429    Size: X-LargeColor: BeigeHeight: 5'9"Weight: 1...
Name: productVariant, dtype: str

`productVariant` is a semi-structured free-text field packing multiple attributes
(Size/Color/Height/Weight) into one string with no consistent delimiter or field order between
rows — will need parsing logic in the Transformer if any of these sub-attributes are wanted
individually, not just a null/not-null check. ~7.8% null, consistent with products that simply
don't have variants.

In [21]:
reviews["reviewMetadata"].dropna().sample(10, random_state=2)

3155    Reviewed in the United States on October 30, 2024
954     Reviewed in the United States on December 21, ...
5565    Reviewed in the United States on January 18, 2025
2502    Reviewed in the United States on January 16, 2025
231     Reviewed in the United States on December 20, ...
1527    Reviewed in the United States on January 23, 2025
15         Reviewed in the United States on March 8, 2025
2409    Reviewed in the United States on December 3, 2024
4575    Reviewed in the United States on February 14, ...
3308    Reviewed in the United States on February 19, ...
Name: reviewMetadata, dtype: str

`reviewMetadata` packs a location and a date into one string (`"Reviewed in <location>
on <date>"`) with no separator — will need parsing (regex or split) to extract structured
`location` and `date` fields in the Transformer, rather than keeping this as one opaque
string.

## 7. Candidate Questions

Just what the data suggests is answerable.

- Does `verifiedPurchase` correlate with `rating` or `sentiment_score`?
- Does having attached images/video correlate with `helpfulVoteCount`?
- How does `sentiment_score` distribution differ between English and non-English reviews
  (once that's identifiable — see finding above)?
- Which products have the widest spread between `rating` and `sentiment_score` (i.e. star
  rating and text sentiment disagree)?
- Does `reviewPosition` (1–10) correlate with `helpfulVoteCount` — do earlier-shown reviews
  get more helpful votes regardless of content (regardless of the stale data of that column)?